[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/A.%20Foundational%20Analytics%20%26%20Inventory%20Control%20%28The%20Language%20of%20Supply%20Chain%29/050.%20The%20New%20Product%20Forecasting%20Problem/P50-Tier-3_executed.ipynb)

---


# 50. The New Product Forecasting Problem: Look-Alike Modeling

## Tier 3: A Krill Herd Metaheuristic

### Purpose and scope

This notebook uses a population-based search to select historical products (analogs) and assign them nonnegative weights for forecasting a new product across market segments and periods. Each individual in the population encodes a candidate analog selection and a set of weights. The Krill Herd search updates these candidates through neighbor-induced motion, movement toward the current best candidate, and random diffusion. The implementation also applies blend crossover and mutation as additional population operators.

The input consists of historical sales arrays, similarity scores between each historical product and the new product, and market-segment adjustment factors. The output is a selected analog set, normalized weights, segment forecasts, and search diagnostics. The method is stochastic and does not guarantee a globally optimal solution.

### Model assumptions

- Historical sales values are comparable across products and periods. In the example, they are interpreted as thousands of units.
- A product's similarity score is a fixed multiplier on its historical sales contribution.
- Each segment's demand is obtained by multiplying the weighted analog forecast by that segment's size and growth factors.
- The fitness function is a temporal-variance proxy with a penalty for using more analogs. It does not compare forecasts with observed future demand, so it is not a statistical accuracy measure.
- The search permits between one and `max_analogs` selected products. The implementation does not impose additional business constraints, such as minimum order quantities, capacity limits, or segment-specific analog sets.

### How to read the search

A krill's position is a real-valued vector. Its first half is decoded into binary selection decisions using a threshold of 0.5; its second half supplies nonnegative weight values, normalized over the selected products so that their weights sum to one. A position is therefore a search representation, while the decoded selection and weights define the forecast candidate evaluated by the fitness function.

For a candidate with selected analog set $A$, normalized weights $w_i$, market segment $s$, and period $t$, the forecast is

$$
\widehat{D}_{s,t} = f_s g_s \sum_{i \in A} w_i q_i x_{i,t},
$$

where $x_{i,t}$ is the historical sales of analog $i$ in period $t$, $q_i$ is its similarity score, and $f_s$ and $g_s$ are the segment size and growth factors. In the code, this calculation is performed by `KrillHerdOptimizer._calculate_segment_forecast`.

The fitness minimized by the search is

$$
F(A,w) = \sum_{s \in S} P\,\operatorname{Var}_{t=1,\ldots,P}(\widehat{D}_{s,t}) + 0.1|A|,
$$

where $S$ is the set of market segments, $P$ is the number of forecast periods, and $|A|$ is the number of selected analogs. The factor $P$ makes each segment's variance contribution equal to its sum of squared deviations from its own temporal mean. The second term penalizes larger analog sets. The code implements these terms in `_evaluate_fitness`. Because no observed future demand is supplied, a lower fitness means a smoother forecast under this proxy and a smaller complexity penalty; it does not establish greater real-world forecasting accuracy.

### Search steps

1. **Initialize the population.** Generate bounded random positions and initial velocities for the krill individuals.
2. **Evaluate candidates.** Decode each position, calculate segment forecasts, and evaluate its fitness.
3. **Calculate neighbor-induced motion.** Among the five closest other individuals, better candidates within the influence range attract the current individual.
4. **Calculate foraging motion.** Direct the individual toward the best candidate in the current population.
5. **Add random diffusion.** Introduce a bounded random direction to encourage exploration.
6. **Update positions.** Combine inertia and the three movement components, limit speed, and clip positions to the interval $[0,1]$.
7. **Apply genetic operators.** Blend crossover and mutation can replace an individual when the resulting candidate has lower fitness.
8. **Track progress.** Record best-so-far fitness and population diversity, and stop early if the best fitness changes by less than the specified tolerance over the recent-history window.

In this implementation, the population size is $N$, the position dimension is $d=2n$ for $n$ historical products, and the number of iterations is at most $T$. Evaluating fitness costs $O(SAP)$ per candidate for $S$ segments and at most $A$ selected analogs. Neighbor distances require $O(N^2d)$ work per iteration, in addition to fitness evaluation and neighbor sorting. Thus, the main search cost grows approximately as $O(T(N^2d+NSAP))$. This describes the implementation's work; it is not an optimality guarantee.

### Example configuration

The TechNova example contains eight historical smartphone models, four market segments, and six forecast periods. The search uses a population of 50 individuals and allows up to 200 iterations. Its motion inertia, foraging inertia, and diffusion factor are 0.9, 0.2, and 0.1, respectively. Crossover rate, mutation rate, and maximum speed are also specified in the configuration cell. The random seed used to generate the example and initialize the search is 42.

The notebook reports the solution and diagnostics produced when its code is run. Results can vary if the seed or stochastic search behavior changes. No comparison with other algorithms or claim of forecast accuracy is made here, because the notebook does not provide common validation data or reproducible benchmark runs for such a comparison.

### Interpreting the diagnostics

- **Best-fitness history** records the lowest fitness found up to each iteration; a flat curve indicates no recent improvement under this objective.
- **Population diversity** is the mean Euclidean distance of individuals' position vectors from their population mean. It describes variation in the encoded search positions, not necessarily variation in decoded analog sets.
- **Final-population fitness summaries** describe the candidates remaining at the end of the search. They need not all match the best-so-far candidate.
- **Analog selection frequency** counts how often each product appears in the final population. It is a descriptive search diagnostic, not a probability that the analog is correct.

The metaheuristic is useful as a way to explore a nonconvex, mixed selection-and-weight search space when a practical solution is sought without an optimality guarantee. Its effectiveness depends on the objective, the encoding, and the algorithm parameters; the marine-behavior metaphor alone does not establish search quality.

In [ ]:
from dataclasses import dataclass
from typing import List, Optional
import time

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

plt.style.use("default")
sns.set_palette("husl")

In [ ]:
@dataclass
class MarketSegment:
    """Demand adjustment factors for one market segment."""

    name: str
    index: int
    size_factor: float
    growth_factor: float


@dataclass
class KrillIndividual:
    """One encoded candidate in the population."""

    position: np.ndarray
    fitness: float
    velocity: np.ndarray
    selected_analogs: List[int]
    weights: List[float]


@dataclass
class KrillHerdParameters:
    """Parameters controlling the Krill Herd search and its operators."""

    population_size: int = 50
    max_iterations: int = 200
    motion_inertia: float = 0.9
    foraging_inertia: float = 0.2
    diffusion_factor: float = 0.1
    crossover_rate: float = 0.8
    mutation_rate: float = 0.05
    max_speed: float = 0.1


@dataclass
class MetaheuristicResult:
    """Solution and diagnostic histories returned by the search."""

    best_solution: KrillIndividual
    convergence_history: List[float]
    diversity_history: List[float]
    best_fitness_history: List[float]
    computation_time: float
    iterations_converged: int
    final_population: List[KrillIndividual]

In [ ]:
class KrillHerdOptimizer:
    """Krill Herd search for analog selection and weight allocation."""

    def __init__(
        self,
        products,
        similarities,
        market_segments,
        max_analogs=3,
        params=None,
    ):
        self.products = [np.asarray(product, dtype=float) for product in products]
        self.similarities = np.asarray(similarities, dtype=float)
        self.market_segments = list(market_segments)
        self.max_analogs = int(max_analogs)

        if not self.products:
            raise ValueError("At least one historical product is required.")
        if len(self.similarities) != len(self.products):
            raise ValueError("Each product must have exactly one similarity score.")
        if not self.market_segments:
            raise ValueError("At least one market segment is required.")
        if any(product.ndim != 1 for product in self.products):
            raise ValueError("Each product's historical sales must be one-dimensional.")
        if len({len(product) for product in self.products}) != 1:
            raise ValueError("All products must have the same number of periods.")
        if not 1 <= self.max_analogs <= len(self.products):
            raise ValueError("max_analogs must be between 1 and the number of products.")

        self.n_products = len(self.products)
        self.n_segments = len(self.market_segments)
        self.n_periods = len(self.products[0])
        self.params = params if params is not None else KrillHerdParameters()

        if self.params.population_size < 1:
            raise ValueError("population_size must be at least 1.")
        if self.params.max_iterations < 1:
            raise ValueError("max_iterations must be at least 1.")

        # One coordinate per selection decision and one per analog weight.
        self.position_dim = self.n_products * 2

    def optimize(self) -> MetaheuristicResult:
        """Run the population search and return its best candidate and history."""
        start_time = time.time()
        population = self._initialize_population()
        convergence_history = []
        diversity_history = []
        best_fitness_history = []
        best_krill = None
        best_fitness = float("inf")

        for iteration in range(self.params.max_iterations):
            for krill in population:
                krill.fitness = self._evaluate_fitness(krill)

            current_best = min(population, key=lambda individual: individual.fitness)
            if current_best.fitness < best_fitness:
                best_fitness = current_best.fitness
                best_krill = self._copy_individual(current_best)

            diversity = self._calculate_diversity(population)

            new_population = []
            for index, krill in enumerate(population):
                motion = self._calculate_motion_induced(krill, population, index)
                foraging = self._calculate_foraging_motion(krill, population)
                diffusion = self._calculate_random_diffusion()

                new_velocity = (
                    self.params.motion_inertia * krill.velocity
                    + self.params.foraging_inertia * (motion + foraging)
                    + self.params.diffusion_factor * diffusion
                )
                speed = np.linalg.norm(new_velocity)
                if speed > self.params.max_speed:
                    new_velocity = new_velocity / speed * self.params.max_speed

                new_position = self._apply_position_bounds(
                    krill.position + new_velocity
                )
                new_krill = self._create_krill_from_position(new_position)
                new_krill.velocity = new_velocity
                new_population.append(new_krill)

            population = self._apply_genetic_operators(new_population)
            convergence_history.append(best_fitness)
            best_fitness_history.append(best_fitness)
            diversity_history.append(diversity)

            # Stop when the best-so-far objective has stabilized over the recent window.
            if iteration > 20 and len(convergence_history) > 10:
                recent_improvement = abs(
                    convergence_history[-1] - convergence_history[-10]
                )
                if recent_improvement < 1e-6:
                    break

        # Evaluate the last updated population so returned diagnostics describe it.
        for krill in population:
            krill.fitness = self._evaluate_fitness(krill)
        final_best = min(population, key=lambda individual: individual.fitness)
        if final_best.fitness < best_fitness:
            best_krill = self._copy_individual(final_best)

        computation_time = time.time() - start_time
        return MetaheuristicResult(
            best_solution=best_krill,
            convergence_history=convergence_history,
            diversity_history=diversity_history,
            best_fitness_history=best_fitness_history,
            computation_time=computation_time,
            iterations_converged=len(convergence_history),
            final_population=population,
        )

    @staticmethod
    def _copy_individual(krill: KrillIndividual) -> KrillIndividual:
        """Keep a separate best-so-far candidate as the population evolves."""
        return KrillIndividual(
            position=krill.position.copy(),
            fitness=krill.fitness,
            velocity=krill.velocity.copy(),
            selected_analogs=list(krill.selected_analogs),
            weights=list(krill.weights),
        )

    def _initialize_population(self) -> List[KrillIndividual]:
        """Create random bounded positions and small initial velocities."""
        population = []
        for _ in range(self.params.population_size):
            position = np.random.random(self.position_dim)
            krill = self._create_krill_from_position(position)
            krill.velocity = np.random.random(self.position_dim) * 0.01
            population.append(krill)
        return population

    def _create_krill_from_position(self, position: np.ndarray) -> KrillIndividual:
        """Decode a continuous position into a feasible analog set and weights."""
        position = np.asarray(position, dtype=float)
        if position.shape != (self.position_dim,):
            raise ValueError("Position vector has the wrong dimension.")

        selection_part = position[:self.n_products]
        weight_part = position[self.n_products:]
        binary_selection = (selection_part > 0.5).astype(int)
        selected_indices = np.flatnonzero(binary_selection)

        if len(selected_indices) > self.max_analogs:
            # Rank only selected candidates, preserving their product indices.
            selected_indices = selected_indices[
                np.argsort(self.similarities[selected_indices])[-self.max_analogs:]
            ]
            binary_selection = np.zeros(self.n_products, dtype=int)
            binary_selection[selected_indices] = 1
        elif len(selected_indices) == 0:
            best_index = int(np.argmax(self.similarities))
            binary_selection[best_index] = 1
            selected_indices = np.array([best_index])

        selected_weight_values = weight_part[selected_indices]
        weight_sum = float(np.sum(selected_weight_values))
        if weight_sum <= np.finfo(float).eps:
            weights = np.full(len(selected_indices), 1.0 / len(selected_indices))
        else:
            weights = selected_weight_values / weight_sum

        return KrillIndividual(
            position=position.copy(),
            fitness=0.0,
            velocity=np.zeros(self.position_dim),
            selected_analogs=selected_indices.tolist(),
            weights=weights.tolist(),
        )

    def _evaluate_fitness(self, krill: KrillIndividual) -> float:
        """Evaluate temporal forecast variation plus the analog-count penalty."""
        total_error = 0.0
        for segment in self.market_segments:
            segment_forecast = self._calculate_segment_forecast(
                krill.selected_analogs, krill.weights, segment
            )
            total_error += np.var(segment_forecast) * len(segment_forecast)

        complexity_penalty = 0.1 * len(krill.selected_analogs)
        return float(total_error + complexity_penalty)

    def _calculate_segment_forecast(self, analogs, weights, segment):
        """Combine weighted analog sales and apply the segment's adjustment factors."""
        forecast = []
        for period in range(self.n_periods):
            period_demand = 0.0
            for index, analog_index in enumerate(analogs):
                similarity = self.similarities[analog_index]
                historical_sales = self.products[analog_index][period]
                period_demand += weights[index] * historical_sales * similarity

            period_demand *= segment.size_factor * segment.growth_factor
            forecast.append(period_demand)
        return forecast

    def _calculate_motion_induced(
        self,
        krill: KrillIndividual,
        population: List[KrillIndividual],
        index: int,
    ) -> np.ndarray:
        """Move toward better nearby individuals among the closest neighbors."""
        motion = np.zeros(self.position_dim)
        neighbor_distances = []
        for other_index, other in enumerate(population):
            if other_index != index:
                distance = np.linalg.norm(krill.position - other.position)
                neighbor_distances.append((distance, other))

        neighbor_distances.sort(key=lambda item: item[0])
        for distance, neighbor in neighbor_distances[:min(5, len(neighbor_distances))]:
            if distance < 1.0 and neighbor.fitness < krill.fitness:
                influence = np.exp(-distance) / (distance + 1e-6)
                motion += influence * (neighbor.position - krill.position)
        return motion

    def _calculate_foraging_motion(
        self,
        krill: KrillIndividual,
        population: List[KrillIndividual],
    ) -> np.ndarray:
        """Direct movement toward the current population's best candidate."""
        best_krill = min(population, key=lambda individual: individual.fitness)
        foraging_direction = best_krill.position - krill.position
        distance = np.linalg.norm(foraging_direction)

        if distance == 0:
            return np.zeros(self.position_dim)

        foraging_motion = foraging_direction / distance
        # The epsilon keeps the scale defined when either fitness is zero.
        fitness_scale = (best_krill.fitness + 1e-6) / (krill.fitness + 1e-6)
        foraging_motion *= min(1.0, fitness_scale)
        return foraging_motion

    def _calculate_random_diffusion(self) -> np.ndarray:
        """Generate a random direction in each coordinate for exploration."""
        return np.random.random(self.position_dim) * 2 - 1

    @staticmethod
    def _apply_position_bounds(position: np.ndarray) -> np.ndarray:
        """Keep the continuous search representation inside its unit bounds."""
        return np.clip(position, 0, 1)

    def _apply_genetic_operators(
        self,
        population: List[KrillIndividual],
    ) -> List[KrillIndividual]:
        """Apply blend crossover and mutation, retaining improvements only."""
        new_population = list(population)

        for index in range(len(population)):
            if np.random.random() < self.params.crossover_rate:
                partner_index = int(np.random.randint(len(population)))
                partner = population[partner_index]
                alpha = np.random.random()
                child_position = (
                    alpha * population[index].position
                    + (1 - alpha) * partner.position
                )
                child = self._create_krill_from_position(child_position)
                child.velocity = (population[index].velocity + partner.velocity) / 2
                child.fitness = self._evaluate_fitness(child)
                if child.fitness < population[index].fitness:
                    new_population[index] = child

        for index, krill in enumerate(new_population):
            if np.random.random() < self.params.mutation_rate:
                mutation_strength = 0.1
                mutation = (
                    np.random.random(self.position_dim) * mutation_strength * 2
                    - mutation_strength
                )
                mutated_position = self._apply_position_bounds(
                    krill.position + mutation
                )
                mutated_krill = self._create_krill_from_position(mutated_position)
                mutated_krill.velocity = krill.velocity.copy()
                mutated_krill.fitness = self._evaluate_fitness(mutated_krill)
                if mutated_krill.fitness < krill.fitness:
                    new_population[index] = mutated_krill

        return new_population

    @staticmethod
    def _calculate_diversity(population: List[KrillIndividual]) -> float:
        """Measure mean Euclidean distance from the population's position mean."""
        if len(population) <= 1:
            return 0.0
        positions = np.array([krill.position for krill in population])
        center = np.mean(positions, axis=0)
        distances = [np.linalg.norm(position - center) for position in positions]
        return float(np.mean(distances))

    def print_solution_summary(self, result: MetaheuristicResult) -> None:
        """Print the selected analogs, objective, and run diagnostics."""
        print("=" * 70)
        print("KRILL HERD SEARCH RESULTS")
        print("=" * 70)
        print(
            f"Products: {self.n_products}, Segments: {self.n_segments}, "
            f"Periods: {self.n_periods}"
        )
        print(f"Population size: {self.params.population_size}")
        print(f"Maximum iterations: {self.params.max_iterations}")
        print(f"Maximum analogs: {self.max_analogs}")
        print(f"Iterations completed: {result.iterations_converged}")
        print(f"Computation time: {result.computation_time:.2f} seconds")
        print("\nBest candidate:")
        print(f"  Selected analog indices: {result.best_solution.selected_analogs}")
        print(
            "  Normalized weights: "
            f"{[f'{weight:.3f}' for weight in result.best_solution.weights]}"
        )
        print(f"  Fitness: {result.best_solution.fitness:.4f}")
        print("\nSearch parameters:")
        print(f"  Motion inertia: {self.params.motion_inertia}")
        print(f"  Foraging inertia: {self.params.foraging_inertia}")
        print(f"  Diffusion factor: {self.params.diffusion_factor}")
        print(f"  Crossover rate: {self.params.crossover_rate}")
        print(f"  Mutation rate: {self.params.mutation_rate}")
        if result.convergence_history:
            print("\nBest-so-far fitness history:")
            print(f"  First recorded value: {result.convergence_history[0]:.4f}")
            print(f"  Last recorded value: {result.convergence_history[-1]:.4f}")
            print(f"  Final population diversity: {result.diversity_history[-1]:.4f}")

In [ ]:
def create_krill_herd_example():
    """Create the TechNova example data used by the search."""
    np.random.seed(42)
    products = [
        np.array([120, 95, 78, 65, 52, 45]),
        np.array([85, 110, 102, 88, 75, 68]),
        np.array([95, 88, 93, 79, 71, 65]),
        np.array([65, 72, 68, 58, 52, 48]),
        np.array([75, 82, 78, 68, 61, 55]),
        np.array([55, 68, 72, 65, 59, 54]),
        np.array([45, 58, 62, 58, 53, 49]),
        np.array([35, 48, 55, 52, 48, 45]),
    ]

    similarities = np.array([0.87, 0.79, 0.71, 0.64, 0.58, 0.52, 0.47, 0.41])
    market_segments = [
        MarketSegment("Premium", 0, 1.3, 1.15),
        MarketSegment("Upper-mid", 1, 1.1, 1.05),
        MarketSegment("Mid-range", 2, 0.9, 0.95),
        MarketSegment("Budget", 3, 0.7, 0.85),
    ]
    return products, similarities, market_segments


products, similarities, market_segments = create_krill_herd_example()

params = KrillHerdParameters(
    population_size=50,
    max_iterations=200,
    motion_inertia=0.9,
    foraging_inertia=0.2,
    diffusion_factor=0.1,
    crossover_rate=0.8,
    mutation_rate=0.05,
    max_speed=0.1,
)

optimizer = KrillHerdOptimizer(
    products,
    similarities,
    market_segments,
    max_analogs=3,
    params=params,
)
result = optimizer.optimize()
optimizer.print_solution_summary(result)

# Check feasibility and the consistency of the decoded candidate.
assert 1 <= len(result.best_solution.selected_analogs) <= optimizer.max_analogs
assert len(result.best_solution.selected_analogs) == len(result.best_solution.weights)
assert np.isclose(sum(result.best_solution.weights), 1.0)
assert np.isclose(
    result.best_solution.fitness,
    optimizer._evaluate_fitness(result.best_solution),
)

In [ ]:
def create_krill_visualizations(result: MetaheuristicResult, products):
    """Plot search progress, final-population fitness, and analog frequency."""
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle(
        "Krill Herd Search Diagnostics",
        fontsize=16,
        fontweight="bold",
    )

    iterations = range(1, len(result.convergence_history) + 1)
    ax1 = axes[0, 0]
    ax1.plot(iterations, result.convergence_history, color="blue", linewidth=2)
    ax1.set_title("Best-So-Far Fitness")
    ax1.set_xlabel("Iteration")
    ax1.set_ylabel("Fitness (lower is better)")
    ax1.grid(True, alpha=0.3)

    ax2 = axes[0, 1]
    ax2.plot(iterations, result.diversity_history, color="green", linewidth=2)
    ax2.set_title("Population Diversity")
    ax2.set_xlabel("Iteration")
    ax2.set_ylabel("Mean distance from position mean")
    ax2.grid(True, alpha=0.3)

    final_fitness = [krill.fitness for krill in result.final_population]
    fitness_values = [
        min(final_fitness),
        float(np.mean(final_fitness)),
        max(final_fitness),
    ]
    ax3 = axes[1, 0]
    bars = ax3.bar(
        ["Best", "Average", "Worst"],
        fitness_values,
        color=["green", "orange", "red"],
        alpha=0.7,
    )
    ax3.set_title("Final-Population Fitness")
    ax3.set_ylabel("Fitness (lower is better)")
    ax3.grid(True, alpha=0.3, axis="y")
    for bar, value in zip(bars, fitness_values):
        ax3.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            f"{value:.3f}",
            ha="center",
            va="bottom",
        )

    selection_counts = np.zeros(len(products), dtype=int)
    for krill in result.final_population:
        for analog_index in krill.selected_analogs:
            selection_counts[analog_index] += 1

    selected = set(result.best_solution.selected_analogs)
    bar_colors = [
        "red" if index in selected else "lightblue"
        for index in range(len(products))
    ]
    ax4 = axes[1, 1]
    bars = ax4.bar(
        [f"Product {index}" for index in range(len(products))],
        selection_counts,
        color=bar_colors,
        alpha=0.7,
    )
    ax4.set_title("Analog Selection Frequency in Final Population")
    ax4.set_xlabel("Historical product")
    ax4.set_ylabel("Number of individuals")
    ax4.tick_params(axis="x", rotation=45)
    ax4.grid(True, alpha=0.3, axis="y")
    ax4.bar([], [], color="red", label="Selected in best candidate")
    ax4.bar([], [], color="lightblue", label="Not selected in best candidate")
    ax4.legend()
    for bar, count in zip(bars, selection_counts):
        if count > 0:
            ax4.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height(),
                str(count),
                ha="center",
                va="bottom",
            )

    fig.tight_layout()
    plt.show()
    return fig


fig = create_krill_visualizations(result, products)

In [ ]:
def analyze_best_solution_forecasts(
    result: MetaheuristicResult,
    optimizer: KrillHerdOptimizer,
):
    """Calculate, report, and plot forecasts from the best candidate."""
    print("\nDETAILED FORECAST ANALYSIS: BEST CANDIDATE")
    print("=" * 55)
    segment_forecasts = {}

    for segment in optimizer.market_segments:
        forecast = optimizer._calculate_segment_forecast(
            result.best_solution.selected_analogs,
            result.best_solution.weights,
            segment,
        )
        segment_forecasts[segment.name] = forecast
        print(f"\n{segment.name} segment:")
        print(f"  Selected analog indices: {result.best_solution.selected_analogs}")
        print(
            "  Weights: "
            f"{[f'{weight:.3f}' for weight in result.best_solution.weights]}"
        )
        print(
            "  Forecast by period (thousand units): "
            f"{[f'{value:.1f}' for value in forecast]}"
        )
        print(f"  Total across periods: {sum(forecast):.1f} thousand units")
        print(f"  Mean per period: {np.mean(forecast):.1f} thousand units")
        print(f"  Temporal variance: {np.var(forecast):.2f}")

    periods = [f"Period {index + 1}" for index in range(optimizer.n_periods)]
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
    fig.suptitle("Best Candidate: Segment Forecast Analysis", fontweight="bold")

    for segment_name, forecast in segment_forecasts.items():
        ax1.plot(periods, forecast, marker="o", linewidth=2, label=segment_name)
    ax1.set_title("Forecast by Market Segment")
    ax1.set_ylabel("Forecasted demand (thousand units)")
    ax1.set_xlabel("Period")
    ax1.legend(title="Market segment")
    ax1.grid(True, alpha=0.3)
    ax1.tick_params(axis="x", rotation=45)

    total_forecast = np.sum(
        np.array(list(segment_forecasts.values()), dtype=float), axis=0
    )
    bars = ax2.bar(periods, total_forecast, color="purple", alpha=0.7)
    ax2.plot(periods, total_forecast, "ro-", linewidth=2, label="Total forecast")
    ax2.set_title("Total Forecast Across Segments")
    ax2.set_ylabel("Total demand (thousand units)")
    ax2.set_xlabel("Period")
    ax2.legend()
    ax2.grid(True, alpha=0.3, axis="y")
    ax2.tick_params(axis="x", rotation=45)
    for bar, value in zip(bars, total_forecast):
        ax2.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            f"{value:.1f}",
            ha="center",
            va="bottom",
        )

    fig.tight_layout()
    plt.show()

    total_horizon_forecast = float(np.sum(total_forecast))
    print("\nAggregate forecast summary:")
    print(f"  Total across all periods: {total_horizon_forecast:.1f} thousand units")
    print(
        "  Average per period: "
        f"{np.mean(total_forecast):.1f} thousand units"
    )
    print(
        f"  Highest period: {periods[int(np.argmax(total_forecast))]} "
        f"({np.max(total_forecast):.1f} thousand units)"
    )
    print(
        f"  Lowest period: {periods[int(np.argmin(total_forecast))]} "
        f"({np.min(total_forecast):.1f} thousand units)"
    )
    print(
        "  Standard deviation across periods: "
        f"{np.std(total_forecast):.2f} thousand units"
    )
    return segment_forecasts, total_forecast


segment_forecasts, total_forecast = analyze_best_solution_forecasts(result, optimizer)

In [ ]:
def report_search_diagnostics(result: MetaheuristicResult):
    """Summarize observed diagnostics from this Krill Herd run."""
    final_fitness = [krill.fitness for krill in result.final_population]
    diagnostics = {
        "best-so-far fitness": result.best_solution.fitness,
        "final population mean fitness": float(np.mean(final_fitness)),
        "final population worst fitness": float(np.max(final_fitness)),
        "iterations completed": result.iterations_converged,
        "computation time (seconds)": result.computation_time,
        "final position diversity": result.diversity_history[-1],
    }

    print("SEARCH DIAGNOSTICS")
    print("=" * 40)
    for label, value in diagnostics.items():
        print(f"{label}: {value:.4f}" if isinstance(value, float) else f"{label}: {value}")
    print(
        "\nFitness is the variance-and-complexity objective defined in this notebook; "
        "it is not a measured forecast-accuracy score."
    )
    return diagnostics


search_diagnostics = report_search_diagnostics(result)

In [ ]:
def print_metaheuristic_summary(
    optimizer: KrillHerdOptimizer,
    result: MetaheuristicResult,
    total_forecast,
):
    """Summarize the model, returned candidate, and interpretation limits."""
    print("\n" + "=" * 70)
    print("KRILL HERD LOOK-ALIKE MODEL: RUN SUMMARY")
    print("=" * 70)

    print("\nSearch configuration:")
    print(f"  Population size: {optimizer.params.population_size}")
    print(f"  Maximum iterations: {optimizer.params.max_iterations}")
    print(f"  Products: {optimizer.n_products}")
    print(f"  Market segments: {optimizer.n_segments}")
    print(f"  Forecast periods: {optimizer.n_periods}")
    print(f"  Position dimensions: {optimizer.position_dim}")
    print("  Search components: neighbor motion, foraging, random diffusion")
    print("  Additional population operators: blend crossover and mutation")

    print("\nBest candidate:")
    print(f"  Analog indices: {result.best_solution.selected_analogs}")
    print(f"  Number of analogs: {len(result.best_solution.selected_analogs)}")
    print(
        "  Normalized weights: "
        f"{[f'{weight:.3f}' for weight in result.best_solution.weights]}"
    )
    print(f"  Fitness: {result.best_solution.fitness:.4f}")
    print(f"  Iterations completed: {result.iterations_converged}")
    print(f"  Computation time: {result.computation_time:.2f} seconds")
    print(f"  Final position diversity: {result.diversity_history[-1]:.4f}")
    print(
        "  Total forecast across the horizon: "
        f"{np.sum(total_forecast):.1f} thousand units"
    )

    if len(result.convergence_history) > 1:
        improvement = result.convergence_history[0] - result.convergence_history[-1]
        print("\nBest-so-far objective history:")
        print(f"  First recorded fitness: {result.convergence_history[0]:.4f}")
        print(f"  Last recorded fitness: {result.convergence_history[-1]:.4f}")
        print(f"  Change in recorded fitness: {improvement:.4f}")

    print("\nInterpretation and limitations:")
    print("  - The search has no guarantee of finding a global optimum.")
    print("  - The objective rewards lower temporal variance and penalizes extra analogs.")
    print("  - The objective does not measure accuracy against observed future demand.")
    print("  - Stochastic results depend on the seed and search configuration.")
    print("  - Forecasts inherit the assumptions in the historical data and segment factors.")
    print("=" * 70)


print_metaheuristic_summary(optimizer, result, total_forecast)